# Apache Flume

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache Flume is a distributed, reliable, and available service for efficiently collecting, aggregating, and moving large amounts of log and event data into HDFS, HBase, Kafka, Solr, or another agent. An agent is a JVM process that wires a source to one or more channels and a sink, and the channel is what makes every handoff transactional.

```text
source -> interceptors -> channel selector -> channel (buffer) -> sink -> storage / next agent
```

Key ideas:

+ A source writes events into a channel; it never writes directly to HDFS.
+ A channel is the durability boundary: memory for speed, file or JDBC for survival across restarts.
+ A sink removes an event from the channel only after the destination acknowledges the batch.
+ The unit of transfer is a transaction; reliability comes from commit/rollback across source, channel, and sink.

## What is Flume?

Flume is a distributed, reliable, and available service for efficiently collecting, aggregating, and moving large amounts of log data. It has a simple and flexible architecture based on streaming data flows. It is robust and fault tolerant with tunable reliability mechanisms and many failover and recovery mechanisms, and it uses an extensible data model that allows for online analytic applications.

```text
web/app servers --taildir--> agent --file channel--> HDFS sink --> partitioned log archive
```

The reliability story in one sentence: the source and the sink each run a transaction against the channel, so an event is either committed end to end or replayed from the channel.

Mechanically an agent is one JVM with three groups of threads. Source threads pull from the outside world and call `channel.put(event)` inside a transaction. Sink threads call `channel.take()` for a batch and commit only after the destination acknowledges the write. A rollback returns the batch to the channel, so the next attempt re-delivers the same events instead of losing them.

Worked example. A source with `batchSize = 500` and a channel with `transactionCapacity = 500` commits one batch per 500 events; at 20,000 events/s that is 40 transactions per second. Raise `transactionCapacity` to 5,000 and the same load is 4 large transactions per second, which is far cheaper in fsyncs for a file channel.

```properties
a1.sources.r1.batchSize = 500
a1.channels.c1.capacity = 200000
a1.channels.c1.transactionCapacity = 5000
```

Common mistake: reading "reliable" as "exactly once". Flume is at-least-once. When the destination write succeeds but the acknowledgement is lost, the sink re-sends the batch. Make the destination idempotent, for example a deterministic file name or an upsert on a business key, instead of expecting Flume to deduplicate.

## Where does Apache Flume fit in modern data platforms?

Flume is used for reliably collecting, aggregating, and moving log and event data into storage systems such as HDFS or HBase. In modern stacks Kafka, cloud-native streaming services, and lightweight collectors are more common, but Flume remains a useful model for source-channel-sink pipeline design and is still found in Hadoop-era clusters and edge aggregation layers.

Example flow:

```text
web server logs -> Flume source -> memory/file channel -> HDFS sink
```

The important concept is the durable handoff between components. Channels buffer events so that a short downstream failure does not immediately lose data.

Flume still pays off where the destination is HDFS or HBase and the environment is already Hadoop-centric. The valuable part is the durable local buffer next to each source, which a shared Kafka cluster does not give an edge host that loses its network for an hour. The channel turns a downstream outage into latency instead of data loss.

Worked example. Four hundred hosts produce 3 TB/day, which is about 35 MB/s across the fleet. A 20 GB file channel per host buffers roughly ten minutes of that host's own output, so the fleet as a whole absorbs about two and a half days of production before sources block. Size the edge buffer for the collector outage you must survive, not for steady state.

Common mistake: replacing Kafka with Flume on the assumption that a channel is a replayable log. A channel is a bounded per-agent queue with no consumer groups, no offsets, and no retention once a sink has drained it. If two teams need the same events later, land them in Kafka or object storage first.

## How do Flume memory channels and file channels differ?

A memory channel is fast but loses buffered events if the agent JVM dies. A file channel is slower but durable because events and checkpoints are written to disk before delivery. Choose based on whether throughput or durability matters more.

+ memory channel: high-volume ephemeral logs where a small loss is acceptable
+ file channel: audit, security, or billing logs where loss is unacceptable
+ JDBC channel: durable and transactional via an embedded Derby database, but rarely the fastest or the most commonly deployed today

For critical pipelines, file channels plus careful disk sizing and monitoring of channel fill percentage are safer.

```properties
# File channel: durable, survives agent restart
a1.channels.c1.type = file
a1.channels.c1.checkpointDir = /data/flume/checkpoint
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.capacity = 1000000
a1.channels.c1.transactionCapacity = 10000
a1.channels.c1.checkpointInterval = 30000
a1.channels.c1.keep-alive = 3
```

The memory channel is a `LinkedBlockingDeque<Event>` on the JVM heap. `capacity` is the number of events it holds, and `byteCapacity` caps total bytes so a few large events cannot exhaust the heap. `transactionCapacity` is the largest batch a source or sink may commit in one transaction.

The file channel appends events to data files under `dataDirs` and records how far each sink has read in `checkpointDir`. It fsyncs on commit by default (`fsyncPerTransaction = true`), which is why it is roughly an order of magnitude slower than memory but survives `kill -9`. `checkpointInterval` bounds how much replay is needed after an unclean shutdown.

```properties
a1.channels.mem.type = memory
a1.channels.mem.capacity = 1000000
a1.channels.mem.transactionCapacity = 20000
a1.channels.mem.byteCapacityBufferPercentage = 20

a1.channels.file.type = file
a1.channels.file.checkpointDir = /data/flume/checkpoint
a1.channels.file.dataDirs = /data/flume/data
a1.channels.file.fsyncPerTransaction = true
a1.channels.file.checkpointInterval = 30000
```

Common mistake: setting `transactionCapacity` equal to `capacity`. While one transaction is in flight the channel has no headroom, and sources and sinks log `ChannelException: Unable to put batch on required channel`. Keep `transactionCapacity` between 1% and 5% of `capacity`.

## What is Apache Flume?

Apache Flume is a distributed, reliable, and available system for efficiently collecting, aggregating, and moving large amounts of log data from many different sources to a centralized store such as HDFS or HBase. Flume is a framework for populating Hadoop with data: agents are deployed across the IT infrastructure - inside web servers, application servers, and network devices - to collect data and integrate it into Hadoop.

A common historical example is Mozilla collecting browser telemetry with Flume into Hive for analysis. The same design still appears as an edge aggregation tier that forwards into Kafka.

Flume is source-channel-sink wiring, and every deployment is really a topology question: where events get buffered, and what happens to that buffer when the next hop dies. The answer is always a channel with a bound, either heap or disk, plus back-pressure upstream when it fills.

Worked numbers. A collector writing 3 TB/day to HDFS needs about 35 MB/s sustained. If the HDFS sink batch is 1,000 events of 1 KB, that is 1 MB per transaction and about 35 transactions per second, which one sink thread handles easily. The file channel's `transactionCapacity` must be at least 1,000 or the sink can never drain a full batch.

```properties
a1.sinks.k1.type = hdfs
a1.sinks.k1.hdfs.batchSize = 1000
a1.channels.c1.transactionCapacity = 2000
```

Common mistake: putting a memory channel on the collector that a whole fleet feeds. One collector JVM restart drops everything buffered in heap while the edge channels keep forwarding. Use a file channel anywhere the events have already left their durable source.

## Which is the reliable channel in Flume to ensure that there is no Data Loss?

The FILE channel is the most reliable of the built-in channels (JDBC, FILE, MEMORY). The file channel writes each event to disk and keeps a checkpoint, so a restart replays anything that was not acknowledged by the sink.

```properties
a1.channels.c1.type = file
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.checkpointDir = /data/flume/checkpoint
```

Reliability also depends on the source and sink semantics: a durable channel cannot make an unreliable source or a non-transactional sink exactly once, it can only guarantee that committed events are not lost.

The file channel is the durable default: events are written to disk and the checkpoint records commit progress, so a restart replays whatever the sink never acknowledged. The JDBC channel is also durable but serializes through Derby and is far slower. The Kafka channel is durable in a topic and is the best of the three when Kafka already exists and you want the buffer to outlive the agent.

Worked example. With `capacity = 5000000`, `transactionCapacity = 20000`, and a 1 KB average event, the channel holds about 4.8 GB and commits 20 MB per transaction. A sink draining at 50 MB/s needs three transactions per second, well inside one sink thread.

Common mistake: claiming a durable channel guarantees no loss regardless of the source. An `exec` source or a `netcat` source has no replay, so events that arrive while the channel is full are lost at the source. Durability starts at the source, not at the channel.

## How can Flume be used with HBase?

Apache Flume can write to HBase through one of two sinks:

+ HBaseSink (`org.apache.flume.sink.hbase.HBaseSink`) supports secure HBase clusters and the newer HBase IPC introduced in HBase 0.96.
+ AsyncHBaseSink (`org.apache.flume.sink.hbase.AsyncHBaseSink`) has better performance because it makes non-blocking calls to HBase.

How HBaseSink works:

+ A Flume event is converted into HBase increments or puts by an `HBaseEventSerializer` that is instantiated when the sink starts.
+ For every event the sink calls the serializer, which translates the event into increments and puts sent to the HBase cluster.

How AsyncHBaseSink works:

+ AsyncHBaseSink implements `AsyncHBaseEventSerializer`; `initialize` is called once when the sink starts.
+ The sink calls `setEvent`, then `getIncrements` and `getActions`, and calls `cleanUp` when it stops.

```properties
# HBase sink with a regex serializer that extracts the row key from the body
a1.sinks.k1.type = hbase
a1.sinks.k1.table = metrics
a1.sinks.k1.columnFamily = d
a1.sinks.k1.serializer = org.apache.flume.sink.hbase.RegexHbaseEventSerializer
a1.sinks.k1.serializer.regex = ^([^|]+)\|([^|]+)\|(.*)$
a1.sinks.k1.serializer.colNames = host,metric,value
a1.sinks.k1.batchSize = 100
a1.sinks.k1.channel = c1
```

HBaseSink creates one HTable client, calls the serializer for each event, and batches the resulting Puts through the HBase client. AsyncHBaseSink uses the non-blocking AsyncHBase client, so a slow region does not stall the sink thread. Both commit to the channel only after HBase acknowledges.

Worked config. With `batchSize = 500` and a regex serializer that extracts `user_id,campaign_id,ts,payload`, 25,000 events/s becomes 50 batches per second. AsyncHBaseSink can take a larger `batchSize` because its calls do not block, but watch the client-side queue depth and the number of region servers you are spreading across.

Common mistake: a regex that does not match the payload. The serializer then emits an empty Put, so the row lands with no columns, or it throws and the sink rolls back on every batch forever. Test the regex against several real events, including a malformed one, before enabling the pipeline.

## What is an Agent?

An agent is a process that hosts Flume components such as sources, channels, and sinks, and therefore has the ability to receive, store, and forward events to their destination. Every agent has its own name and its own configuration block (for example `a1.sources`, `a1.channels`, `a1.sinks`).

```bash
# Start one agent named a1 with a properties config file
flume-ng agent --conf /etc/flume/conf --conf-file /etc/flume/conf/weblogs.conf -n a1 \
  -Dflume.root.logger=INFO,console
```

An agent is the unit of deployment: one JVM, one `-n` name, one block of properties. Components are wired by name lists, so `a1.sources = r1`, `a1.channels = c1`, and `a1.sinks = k1` must agree with the property prefixes or the agent refuses to start.

Every config has the same skeleton, and every skeleton has the same three links: the component lists, `r1.channels = c1` on the source, and `k1.channel = c1` on the sink. Missing any one of the three is the most common startup error in a new agent.

```properties
a1.sources = r1
a1.channels = c1
a1.sinks = k1

a1.sources.r1.type = netcat
a1.sources.r1.bind = 0.0.0.0
a1.sources.r1.port = 44444
a1.sources.r1.channels = c1

a1.channels.c1.type = memory
a1.channels.c1.capacity = 100000

a1.sinks.k1.type = logger
a1.sinks.k1.channel = c1
```

Common mistake: running source, channel, and sink in one JVM and assuming the JVM is not a failure domain. An out-of-memory error in the source takes the sink with it. On a busy host, separate the collection agent from the delivery agent so a slow destination cannot stall log collection.

## Is it possible to Leverage Real Time Analysis on the Big Data collected by Flume directly?

Yes. Data from Flume can be extracted, transformed, and loaded in real time into Apache Solr using `MorphlineSolrSink`, or written into a low-latency store such as HBase for key lookups. For true streaming analytics, the more common pattern is to land events in Kafka and let a stream processor such as Flink or Spark Structured Streaming consume them.

```text
Flume -> MorphlineSolrSink -> Solr index -> search and faceted queries
```

Not really: Flume is a transport and buffering layer, not a query engine. It can deliver in near real time into a store that answers queries, such as Solr through `MorphlineSolrSink`, HBase for lookups, or Kafka for a stream processor, but the analysis happens in that store or processor.

Worked path: `Flume -> Kafka -> Flink` gives seconds-level latency because the Kafka sink commits once the broker acknowledges with `acks=all`, and Flink reads the topic independently of Flume. Chaining `Flume -> Solr` gives index freshness in seconds but no joins or aggregation.

Common mistake: trying to do windowed aggregation with an interceptor. Interceptors are per-event and carry no state across events, so a `regex_extractor` can enrich but cannot count over a window. Put the window in Flink, Spark, or the destination store.

## What is a Channel?

A channel stores events. Events are delivered to the channel by sources running inside the agent, and an event stays in the channel until a sink removes it for further transport. It decouples producers from consumers and absorbs downstream outages.

A channel is a transactional queue between source and sink. `put` from a source and `take` from a sink each run in a transaction; a sink commits only when the destination acknowledges, and a rollback puts the batch back. That commit-or-rollback pair is the whole reliability mechanism in Flume.

The channel also enforces the batch contract. `transactionCapacity` sets the maximum events per transaction, so a sink whose `batchSize` is larger can never drain a full batch. `keep-alive` is how long a sink waits for a batch to fill before sending a partial one, trading latency for throughput.

```properties
a1.channels.c1.keep-alive = 3
a1.sinks.k1.batchSize = 2000
a1.channels.c1.transactionCapacity = 2000
```

Common mistake: leaving `keep-alive` large on a low-volume source. The sink waits the whole interval before sending a half-empty batch, so dashboards see data minutes late. Lower it for low-rate flows and raise it only when batches actually fill.

## Explain about the different channel types in Flume and which channel type is faster?

The built-in channel types are:

+ MEMORY channel - events are held in an in-memory queue and passed to the sink; fastest, with the risk of data loss on agent failure.
+ JDBC channel - events are stored in an embedded Derby database; durable but slow for very high throughput.
+ FILE channel - events are written to files on the local filesystem after the source reads them, and are deleted only after successful delivery to the sink.
+ KAFKA channel - events are stored in a Kafka topic, giving durability plus the ability to serve multiple consumers; a good bridge between Flume collection and a Kafka-centric platform.
+ SPILLABLE MEMORY channel - memory first, spilling to disk when a threshold is reached; a middle ground between memory and file.

MEMORY is the fastest channel, but it has the risk of data loss. The right choice depends on the value of each event and the acceptable recovery path.

| Channel | Speed | Durability | Typical use |
|---|---|---|---|
| memory | highest | none across restart | high-volume, loss-tolerant logs |
| file | medium | high | audit, billing, security logs |
| jdbc | low | high | small legacy pipelines |
| kafka | medium-high | high | decoupled multi-consumer ingestion |
| spillable memory | high until spill | medium | bursty traffic with a disk safety net |

A memory channel is a `LinkedBlockingDeque` on the heap. A file channel is an append-only data log plus a checkpoint. A JDBC channel is a Derby-backed transactional queue. A Kafka channel is a topic that acts as the queue, so the sink commits an offset rather than a local transaction. A spillable memory channel keeps events on heap first and spills to a file channel once `capacity` is reached, up to `overflowCapacity`.

Worked numbers for one stream of 1 KB events at 50,000 events/s. A memory channel handles it in one agent with a few GB of heap. A file channel on a single spinning disk fsync-throttles far below that unless `fsyncPerTransaction = false` or the disk is NVMe. A Kafka channel moves the durability to the brokers and scales by partition. Match the channel to the failure you must survive.

Common mistake: choosing the spillable memory channel as a free upgrade. Events still on heap die with the JVM, only the spilled portion is durable, and only that part is restored after a restart. It is a burst absorber, not a durable channel; if loss is unacceptable, use file or Kafka.

## Explain about the replication and multiplexing selectors in Flume?

Channel selectors decide which channels a source writes to. With no selector configured, the default is the replicating selector, which writes the same event to every channel in the source's `channels` list. The multiplexing selector routes each event to one or more channels based on a header value.

```properties
# Replication: every event goes to both channels
a1.sources.r1.channels = c1 c2
a1.sources.r1.selector.type = replicating
```


Replication is used for fan-out to multiple destinations; multiplexing is used to split different event classes cheaply at the source.

The replicating selector writes every event to every channel in the source's `channels` list. Every channel must accept the event or the whole transaction fails, so replication multiplies both storage and sink capacity. It is the default whenever more than one channel is listed and no selector type is set.

The multiplexing selector maps a header value to a channel list. `selector.header` names the header, `selector.mappings.<value>` names the channels, and `selector.default` catches everything else. Events whose header matches no mapping and has no default are dropped, so always set a default.

```properties
a1.sources.r1.channels = c_err c_archive
a1.sources.r1.selector.type = multiplexing
a1.sources.r1.selector.header = level
a1.sources.r1.selector.mappings.ERROR = c_err
a1.sources.r1.selector.mappings.INFO = c_archive
a1.sources.r1.selector.default = c_archive
```

Common mistake: a multiplexing selector with no `default` and a header that is sometimes absent, for example when the source does not always emit `level`. Those events vanish with only a warning. Set a default channel, or add a `static` interceptor that supplies a fallback header before the selector runs.

## Does Apache Flume provide support for third party Plugins?

Yes. Flume has a plug-in based architecture: sources, channels, sinks, serializers, and interceptors are all configurable class names, so custom implementations can load data from external sources and transfer it to external destinations. A custom sink is written by implementing the Sink interface and packaging the class on the agent classpath.

Plugins are class names in the config, loaded from the agent classpath. A source implements `Source` and `Configurable`, a sink implements `Sink`, an interceptor implements `Interceptor`, and a serializer implements `EventSerializer`. Placing a jar under `$FLUME_HOME/plugins.d/<name>/lib` is enough for the agent to find it.

Example layout: `plugins.d/my-plugin/lib/my-plugin.jar`, with bundled third-party jars under `libext` next to it. Flume scans `plugins.d` at startup, so adding a jar needs an agent restart, and a mistyped class name fails immediately with `ClassNotFoundException` in the log.

Common mistake: packing dependencies into the same jar as the plugin instead of using `libext`. Conflicting Guava or SLF4J versions then break the whole agent, not just the plugin. Keep third-party dependencies under `libext` so they are isolated from Flume's own classpath.

## Differentiate between FileSink and FileRollSink?

HDFS File Sink (`hdfs`) writes events into the Hadoop Distributed File System, while File Roll Sink (`file_roll`) stores events in the local filesystem of the agent host, rolling files by size or count.

```properties
# Roll local files every 10 MB, keeping 24 files
a1.sinks.k1.type = file_roll
a1.sinks.k1.sink.directory = /var/flume/roll
a1.sinks.k1.sink.rollInterval = 0
a1.sinks.k1.sink.batchSize = 100
a1.sinks.k1.sink.serializer = text
```

The HDFS sink (`hdfs`) writes to a Hadoop filesystem and honors `rollSize`, `rollCount`, `rollInterval`, and a time-based path such as `%Y/%m/%d/%H`. `file_roll` writes to the local filesystem of the agent host, rolls by `rollInterval`, `rollSize`, or `rollCount`, and needs a `sink.directory` that already exists. `file_roll` is for local staging and debugging, not for Hadoop.

Worked rolling: at 35 MB/s with `rollSize = 134217728` (128 MB) a new file appears every 3.7 seconds, which is thousands of files a day. Raising `rollSize` to 1 GB gives one file every 30 seconds. The useful HDFS file size is roughly 128 MB to 1 GB, so combine a size trigger with a time trigger.

Common mistake: using `file_roll` with `rollInterval = 0` and no other roll condition. The file never closes, so nothing downstream can read it while the directory keeps growing. Always leave at least one roll trigger active.

## Why we are using Flume?

Hadoop developers use Flume to pull data from sources such as social media, web servers, and application logs. It was developed at Cloudera for aggregating and moving very large amounts of data. The primary use is to gather log files from many sources and asynchronously persist them into the Hadoop cluster, with buffering that tolerates downstream outages.

The reasons are concrete: back-pressure, durability at the edge, and first-class HDFS and HBase sinks. Flume reads only as fast as the sink drains, so a bounded buffer protects both the source host and the destination rather than letting the producer overwhelm storage.

Small files are the classic complaint. A 35 MB/s stream with a 30-second roll writes about 1 GB files, but a 1 MB/s stream with the same roll writes 30 MB files, which Hive and Spark both read inefficiently. Tune `rollSize` and `rollInterval` per stream instead of copying one template across every agent.

Common mistake: treating Flume as a scheduler. Flume has no notion of a job window or a completed partition; it streams continuously. If you need "the hour is complete" semantics, derive that downstream or orchestrate the load with Airflow rather than expecting the agent to stop.

## What is Flumeng?

Flume NG is the second-generation Flume agent architecture: a single JVM agent with pluggable sources, channels, sinks, and interceptors. It stores data in HDFS or HBase and replaced the original multi-process "Flume OG" design. Modern Apache Flume documentation describes Flume NG.

Flume NG replaced the multi-process OG design with a single-JVM agent whose source, channel, sink, and interceptor implementations are pluggable. The practical changes are one `flume-ng` command, one agent per process, and the source-channel-sink transaction model that the rest of this notebook assumes.

Migration note: the OG `master`, `collector`, and `agent` node roles no longer exist. A modern deployment expresses the same topology with multi-hop Avro agents, which is why hostname and port properties replace the old node-role configuration.

Common mistake: following an old Flume OG tutorial. A configuration that expects a central master, a collector node type, or `flume-ng node` subcommands will not start on Flume NG. Check that the documentation targets the NG architecture before copying its properties.

## What are the complicated steps in Flume configurations?

Flume processes streaming data, so once started there is no natural stop or end to the process; it flows data from source to sink through an agent continuously. The agent must be told how its components connect, and external sources add their own credentials. For example, downloading from Twitter requires `consumerKey`, `consumerSecret`, `accessToken`, and `accessTokenSecret`.

The steps that cause the most incidents in practice:

+ Matching source, channel, and sink names consistently across the three lists.
+ Sizing a file channel (capacity, transactionCapacity, disk) so it cannot fill up.
+ Choosing the right HDFS sink parameters (`rollSize`, `rollCount`, `round`, `hdfs.fileType`).
+ Handling HDFS `append` vs `create` and small-file accumulation.
+ Restarting an agent without losing the file channel checkpoint.

```bash
# Validate a config without starting the agent
flume-ng agent --conf /etc/flume/conf --conf-file /etc/flume/conf/weblogs.conf -n a1 \
  -Dflume.root.logger=DEBUG,console --dry-run
```

Most incidents come from four places: component names that do not match across the three lists, a file channel that fills because the sink is slower than the source, HDFS sink parameters that produce tiny files, and a restart that loses the checkpoint because data or checkpoint directories moved.

Worked diagnosis order. Read `ChannelFillPercentage` first; a steady climb means the sink is the bottleneck. Compare `EventPutSuccessCount` with `EventTakeSuccessCount`; a widening gap is the same signal. Then check whether a `rollSize` or `rollCount` change silently multiplied the file count.

Common mistake: pointing `dataDirs` and `checkpointDir` at the same disk as the HDFS client staging directory. Contention shows up as sink timeouts, and a full disk corrupts the checkpoint. Separate the disks and alert on free space, not just on channel fill.

## What are Flume core components?

Source, channel, and sink are the core components.

+ A Flume source receives an event from an external source and stores it in one or more channels.
+ A Flume channel temporarily stores the event until it is consumed by a sink; it acts as the repository that decouples the two sides.
+ A Flume sink removes the event from the channel and puts it into an external repository such as HDFS or HBase, or forwards it to the next Flume agent.

Additional components are interceptors (mutate/filter events in flight), channel selectors (route events to channels), and sink processors (load balancing and failover across sinks).

Source, channel, and sink, plus the optional interceptors, channel selectors, and sink processors. A source has one or more channels, a sink has exactly one channel or belongs to a sink group, and the agent is the process that hosts all of them.

The path in order: the source reads and puts, interceptors transform in the source thread, the selector chooses channels, the channel stores, the sink-group processor chooses a sink, the sink takes and writes, then commits. Knowing where each step runs is what makes a bottleneck visible.

Common mistake: putting an expensive regex extractor in the interceptor chain of a high-volume source. Interceptors run once per event in the source thread, so one slow regex caps source throughput. Move heavy parsing to the sink path, to a record-oriented processor downstream, or remove it.

## What are the Data Extraction Tools in Hadoop?

Sqoop transfers data between relational databases and HDFS, while Flume extracts streaming data from sources such as social media and web logs and stores it in HDFS. In a modern stack these are joined by Kafka Connect (database and service connectors), Debezium (CDC), and cloud-native transfer services.

Sqoop moves bulk relational data in and out of HDFS with a JDBC connection and map-only jobs. Flume moves event streams with a persistent agent and a channel. Their modern counterparts are Kafka Connect and Debezium for databases, and Flume, Fluent Bit, or Vector for host and application telemetry.

Worked split: a nightly 2 TB warehouse extract belongs in Sqoop or a Spark JDBC job, because it has a clear start and end and can parallelize by primary-key ranges. A continuous 30,000 events/s application log stream belongs in Flume, because it never ends and needs a buffer next to the producer.

Common mistake: reaching for a polling SQL source to capture database changes. Polling misses deletes, gives no reliable ordering, and adds load to the source database. Use a log-based CDC tool and keep Flume for file and event ingestion.

## Does Flume provide 100 percents reliability to the Data Flow?

Flume provides end-to-end reliability when it is configured with durable components, because of its transactional approach: the source and sink each commit against the channel. "100%" is a design property, not a default - a memory channel or a non-transactional sink breaks it. There is also no end-to-end exactly-once guarantee to arbitrary destinations; the guarantee is at-least-once plus idempotency at the sink.

End-to-end reliability has three conditions: a replayable source, a durable channel, and a transactional or idempotent sink. Flume guarantees that an event committed to the channel is eventually delivered at least once. It cannot guarantee exactly once, and it cannot recover events that the source never handed it.

Worked duplicate math. If 0.1% of transactions lose their acknowledgement and each transaction carries 5,000 events, then in one million transactions about 1,000 are re-delivered, which is roughly 5 million duplicate events. Deduplicate on a business key or accept the duplicates; do not assume they are rare at scale.

Common mistake: putting a memory channel in front of a sink that is not idempotent and calling the pipeline reliable. A restart loses the buffer and a retry can duplicate, so both failure directions are possible. Use a durable channel and an idempotent sink, or state the tolerance explicitly.

## Tell any two Features of Flume?

+ Flume collects, aggregates, and moves large amounts of log data from many different sources to a centralized data store.
+ Flume is not restricted to log data: it can transport network traffic data, social-media data, email messages, and pretty much any event data.
+ It has tunable reliability (memory vs file channel) and supports failover and load balancing across sinks.

Beyond log collection, Flume is a configurable pipeline: interceptors for enrichment and filtering, selectors for routing, sink groups for failover and load balancing, and pluggable serializers for output format. That composability is why it survived as an edge collection tier after Hadoop peaked.

Features that matter operationally: a bounded channel that provides back-pressure, a JSON metrics endpoint for monitoring, and restart-safe file channels with checkpoints. Together they turn a host agent into a component you can reason about during an incident.

Common mistake: answering with only "it collects logs". The differentiators are the transactional channel, back-pressure, and multi-hop Avro topologies. Without those, Flume is just a tail command with a properties file.

## What are Interceptors?

Interceptors sit between a source and a channel and can inspect or modify events in flight. They are configured as a chain, and each interceptor returns the same event or a modified one; dropping an event is done with a channel selector or a filter interceptor.

Common built-in interceptors:

+ `timestamp` - inserts the collector time into a header.
+ `host` - inserts the agent hostname.
+ `static` - inserts fixed key/value headers.
+ `regex_filter` - drops events matching (or not matching) a regex.
+ `regex_extractor` - extracts fields from the body into headers.
+ `uuid` - adds a random UUID header.

```properties
a1.sources.r1.interceptors = i1 i2 i3
a1.sources.r1.interceptors.i1.type = timestamp
a1.sources.r1.interceptors.i2.type = host
a1.sources.r1.interceptors.i2.preserveExisting = false
a1.sources.r1.interceptors.i3.type = regex_filter
a1.sources.r1.interceptors.i3.regex = ^.*ERROR.*$
a1.sources.r1.interceptors.i3.excludeEvents = false
```

Interceptors are invoked in order by the source before the event reaches a channel. An interceptor builder creates each one, `intercept(event)` returns the event possibly modified, and the batch method can return a shorter list, which is how `regex_filter` drops events.

Order matters. A typical chain is `timestamp`, then `host`, then `static`, then `regex_extractor`, then `regex_filter`. If the filter runs before the extractor, the extractor never sees the dropped events. If a multiplexing selector depends on a header, that header must be set by an interceptor that runs before the selector. Worked config with a static fallback header, so the selector always has a value:

```properties
a1.sources.r1.interceptors = i1 i2
a1.sources.r1.interceptors.i1.type = regex_extractor
a1.sources.r1.interceptors.i1.regex = ^(\\S+)\\s+(.*)$
a1.sources.r1.interceptors.i1.serializers = s1
a1.sources.r1.interceptors.i1.serializers.s1.name = level
a1.sources.r1.interceptors.i2.type = static
a1.sources.r1.interceptors.i2.key = level
a1.sources.r1.interceptors.i2.value = INFO
a1.sources.r1.interceptors.i2.preserveExisting = true
```

Common mistake: getting the `excludeEvents` flag backwards on `regex_filter`. The flag inverts which events survive, so a wrong value silently discards the events you meant to keep. Point the chain at a `logger` sink first and confirm the counts before writing to HDFS.

## Why Flume?

Flume is not limited to collecting logs from distributed systems; it is also capable of other use cases such as:

+ Collecting readings from an array of sensors.
+ Collecting impressions from custom apps for an ad network.
+ Collecting readings from network devices to monitor their performance.

Flume targets reliability, scalability, manageability, and extensibility while serving many clients with high QoS.

Flume's niche is the last mile into Hadoop storage with a buffer that can survive a downstream outage. It is agent-based, so it scales by adding hosts rather than by centralizing, and the file channel turns an HDFS outage into back-pressure instead of loss.

Worked tradeoff: a file channel that fsyncs on every small transaction is slow. Batching 5,000 events per transaction amortizes the fsync and reaches tens of MB/s, but a crash then replays up to `transactionCapacity` events. Reliability versus throughput is exactly this dial.

Common mistake: adding Flume where a lighter shipper would do. For plain file-to-Elasticsearch log shipping, Filebeat or Fluent Bit uses far less memory and has no JVM. Choose Flume when you need HDFS or HBase sinks, or Hadoop-native buffering.

## What is Flume Event?

An event is the unit of data that Flume transports: a byte payload plus a set of string headers. The external source sends events to the source, and internally Flume understands the source format. Each event has a header section (metadata) and a body (the value).

```text
Event {
  headers: { host: web-07, timestamp: 1759600000000 },
  body:    "2026-10-04T13:20:34Z INFO checkout order=88217 amount=42.10"
}
```

Headers are the hook for routing: channel selectors and HDFS sinks read them to decide where an event goes.

An event is `{ headers: Map<String,String>, body: byte[] }`. Headers are small strings copied with the event; the body is the payload and Flume never interprets it. Sources that read text still hand the sink bytes, which is why a serializer or a record processor is needed to change format.

Body size and event count both matter. A `netcat` source creates one event per line, a spooling directory source creates one event per file, and a million 1 KB events behaves very differently from one 1 GB event. A memory channel counts events in `capacity`, so `byteCapacity` is what actually protects the heap.

Common mistake: assuming headers are free. Every header is written into the file channel record and, for sequence files, into the HDFS metadata. Hundreds of headers per event inflate storage and slow the write path, so extract only what routing and partitioning need.

## How Multi hop agent can be setup in Flume?

An Avro RPC bridge is used to set up a multi-hop agent: the first agent ends with an Avro sink, the next agent starts with an Avro source on a port, and the event flows across the network with transaction guarantees at both ends.

```properties
# Agent on the edge: tail local logs, forward over Avro
edge.sources.r1.type = taildir
edge.sources.r1.filegroups = f1
edge.sources.r1.filegroups.f1 = /var/log/app/.*\.log
edge.sources.r1.channels = c1
edge.channels.c1.type = file
edge.channels.c1.checkpointDir = /data/flume/edge/checkpoint
edge.channels.c1.dataDirs = /data/flume/edge/data
edge.sinks.k1.type = avro
edge.sinks.k1.hostname = collector.example.com
edge.sinks.k1.port = 4141
edge.sinks.k1.channel = c1
```

```properties
# Collector agent: receive Avro, write to HDFS
col.sources.r1.type = avro
col.sources.r1.bind = 0.0.0.0
col.sources.r1.port = 4141
col.sources.r1.channels = c1
col.channels.c1.type = file
col.channels.c1.checkpointDir = /data/flume/col/checkpoint
col.channels.c1.dataDirs = /data/flume/col/data
col.sinks.k1.type = hdfs
col.sinks.k1.hdfs.path = hdfs://nn/logs/app/%Y/%m/%d/%H
col.sinks.k1.hdfs.filePrefix = events-
col.sinks.k1.hdfs.fileType = DataStream
col.sinks.k1.hdfs.rollSize = 134217728
col.sinks.k1.hdfs.rollCount = 0
col.sinks.k1.channel = c1
```

Multi-hop uses an Avro RPC bridge. The edge agent's Avro sink takes a batch from its own channel and appends it to the collector's Avro source, which puts it into the collector's channel. The edge sink commits only when the collector's transaction commits, so a collector failure rolls the edge batch back.

Both ends must agree on host, port, and, for secured clusters, Kerberos and TLS settings. The Avro source listens with `bind` and `port`; the Avro sink connects with `hostname` and `port`, and exposes `batchSize`, `connect-timeout`, and `request-timeout` knobs for tuning the RPC.

Common mistake: many edge agents feeding one collector with a memory channel on the collector. A collector restart loses every batch in flight, and the edge channels fill while it is down. Use a file channel on the collector and at least two collectors behind a failover sink group.

## Can Flume can distribute data to multiple destinations?

Yes. Flume supports multiplexing and replication. An event can flow from one source to multiple channels and thus multiple destinations; replication is the default selector behavior when multiple channels are listed.

```properties
a1.sources.r1.channels = c1 c2
a1.sources.r1.selector.type = replicating
a1.sinks.k1.channel = c1
a1.sinks.k2.channel = c2
```

To fan out at the sink layer, a sink group with a load-balancing or failover processor sends each batch to one of several sinks.

```properties
a1.sinkgroups = g1
a1.sinkgroups.g1.sinks = k1 k2
a1.sinkgroups.g1.processor.type = load_balance
a1.sinkgroups.g1.processor.selector = round_robin
a1.sinkgroups.g1.processor.backoff = true
```

There are two fan-out points. At the source, replication or multiplexing writes across channels. At the sink layer, a sink group with a load-balancing or failover processor picks one sink per batch. Source fan-out copies the event to several channels; sink-group fan-out sends each batch to exactly one sink.

Worked failover config with explicit priorities:

```properties
a1.sinkgroups = g1
a1.sinkgroups.g1.sinks = k1 k2
a1.sinkgroups.g1.processor.type = failover
a1.sinkgroups.g1.processor.priority.k1 = 10
a1.sinkgroups.g1.processor.priority.k2 = 5
a1.sinkgroups.g1.processor.maxpenalty = 30000
a1.sinkgroups.g1.processor.backoff = true
```

Common mistake: using load balancing when ordering matters. A load-balancing sink group spreads consecutive batches across sinks, so two events from the same file can land out of order. Use failover for ordered pipelines and load balancing only when the destination is order-insensitive.

## Can you explain about configuration files?

The agent configuration is stored in a local properties file that describes each agent's source, channel, and sink wiring plus their type-specific parameters. The `--conf-file` argument points to it and `-n` names the agent.

A minimal complete agent: spool a directory into a memory channel and log the events.

```properties
# /etc/flume/conf/spool-to-logger.conf
a1.sources = r1
a1.channels = c1
a1.sinks = k1
```


The same file can define several agents; each agent is a separate JVM started with its own `-n` argument.

A config file is a Java properties file. Each agent is a top-level namespace, and inside it live the three component lists plus per-component properties. `--conf-file` points to the file and `-n` selects the agent, so several agents can share one file and run as separate JVMs.

Component types may be fully qualified class names or short aliases such as `memory`, `file`, `hdfs`, `avro`, `kafka`, `logger`, `taildir`, and `spoolDir`. Properties are passed straight to the component, so the authoritative list for a type is its user-guide section, not a copied example.

Common mistake: restarting an agent after an edit without validating the config. One misspelled property can leave a component at a default that is wrong for production, for example a default `capacity` of 100 events. Diff the file against the last known-good version and watch the startup log for warnings.

## How do you write a complete Flume agent configuration?

A production log pipeline needs a directory source, a durable channel, and an HDFS sink whose path is time partitioned. The example below tails application logs, stamps each event, and writes it to HDFS one file per 128 MB or 10 minutes.

```properties
# /etc/flume/conf/weblogs.conf
a1.sources = r1
a1.channels = c1
a1.sinks = k1
```


Design notes:

+ `rollCount = 0` disables count-based rolling so files are governed by size and time, which keeps file sizes predictable.
+ The `year=.../month=...` layout is Hive-friendly partition naming, so the downstream table can use `PARTITIONED BY (year, month, day, hour)`.
+ `rollInterval` must be shorter than the sink `callTimeout` tuning window; otherwise a stalled file accumulates events.
+ The file channel checkpoint directory and data directories should be on separate fast disks with enough free space for at least a few hours of events.

A production config names every component, wires the three links, chooses types, and tunes the channel and sink. The HDFS path should be time-partitioned, the roll policy should target 128 MB to 1 GB files, and the file channel should be sized for the outage window you must survive.

Worked full pipeline: `taildir` source, file channel, HDFS sink, and a host interceptor, writing to time-partitioned directories:

```properties
a1.sources = r1
a1.channels = c1
a1.sinks = k1

a1.sources.r1.type = taildir
a1.sources.r1.positionFile = /var/lib/flume/weblogs_position.json
a1.sources.r1.filegroups = f1
a1.sources.r1.filegroups.f1 = /var/log/app/.*\\.log
a1.sources.r1.channels = c1
a1.sources.r1.interceptors = host
a1.sources.r1.interceptors.host.type = host

a1.channels.c1.type = file
a1.channels.c1.checkpointDir = /data/flume/checkpoint
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.capacity = 2000000
a1.channels.c1.transactionCapacity = 20000

a1.sinks.k1.type = hdfs
a1.sinks.k1.hdfs.path = hdfs://nn/logs/app/year=%Y/month=%m/day=%d/hour=%H
a1.sinks.k1.hdfs.fileType = DataStream
a1.sinks.k1.hdfs.rollSize = 268435456
a1.sinks.k1.hdfs.rollCount = 0
a1.sinks.k1.hdfs.rollInterval = 600
a1.sinks.k1.channel = c1
```

Common mistake: copying an HDFS sink block between streams without revisiting `rollSize` and `rollInterval`. A block tuned for 35 MB/s produces tiny files on a 1 MB/s stream, and NameNode small-file pressure is the usual root cause behind a vague report that "Flume is slow".

## How do you configure a Flume agent to write to Kafka?

A Kafka sink lets Flume hand events to a Kafka topic instead of HDFS, so multiple consumers can read the same stream. A Kafka channel is also available and is durable by itself.

```properties
# Tail logs and publish to Kafka
a1.sources = r1
a1.channels = c1
a1.sinks = k1
```


Reliability notes:

+ `acks=all` means the broker leader waits for in-sync replicas, so a broker failure does not silently drop the event.
+ Keep at least three brokers and a replication factor of three for a topic that a file channel feeds.
+ The Kafka sink is transactional with the channel, so a Kafka outage backs events up in the file channel rather than losing them - size the channel for the expected outage window.

A Kafka sink wraps a `KafkaProducer`. `kafka.bootstrap.servers` points at the brokers, `kafka.topic` names the topic, and any `kafka.producer.*` property passes through to the producer. The sink commits to the channel only when the broker acknowledges according to the `acks` setting, so a broker outage backs events up in the channel instead of losing them.

Worked config and numbers. At 20,000 events/s with `batchSize = 2000` the sink sends 10 batches per second. With `acks=all` on three brokers every acknowledgement waits for the in-sync replicas, so `linger.ms` of 10 to 50 lets batches fill instead of paying a round trip per record.

```properties
a1.sinks.k1.type = org.apache.flume.sink.kafka.KafkaSink
a1.sinks.k1.kafka.bootstrap.servers = broker-1:9092,broker-2:9092,broker-3:9092
a1.sinks.k1.kafka.topic = logs.raw
a1.sinks.k1.kafka.producer.acks = all
a1.sinks.k1.kafka.producer.linger.ms = 20
a1.sinks.k1.kafka.producer.compression.type = lz4
a1.sinks.k1.batchSize = 2000
a1.sinks.k1.channel = c1
```

Common mistake: creating the topic with one partition. Flume can send 2,000 records per batch, but a single partition caps consumer parallelism at one, so downstream can never keep up. Size partitions for the consumers and choose the record key that defines ordering.

## How do you use interceptors in a Flume configuration?

Interceptors are the cheapest place to enrich, filter, and route events. The chain runs in order, and each interceptor sees the output of the previous one.

```properties
a1.sources.r1.interceptors = ts host static extract drop_debug
a1.sources.r1.interceptors.ts.type = timestamp
```


With `level` in a header, a multiplexing selector can send ERROR events to an alerting channel and INFO events to the archive channel without a second parse downstream.

The chain is a space-separated list on `interceptors`, and each element gets `interceptors.<name>.type` plus its own properties. Interceptors belong to the source configuration, so they apply to every event that source reads, before the channel selector sees it.

Worked chain that stamps time, tags the host, extracts a level, and drops debug lines:

```properties
a1.sources.r1.interceptors = ts host extract drop_debug
a1.sources.r1.interceptors.ts.type = timestamp
a1.sources.r1.interceptors.host.type = host
a1.sources.r1.interceptors.extract.type = regex_extractor
a1.sources.r1.interceptors.extract.regex = ^(\\w+)\\s+(.*)$
a1.sources.r1.interceptors.extract.serializers = s1
a1.sources.r1.interceptors.extract.serializers.s1.name = level
a1.sources.r1.interceptors.drop_debug.type = regex_filter
a1.sources.r1.interceptors.drop_debug.regex = ^.*DEBUG.*$
a1.sources.r1.interceptors.drop_debug.excludeEvents = true
```

Common mistake: assuming an interceptor error stops the agent. Most interceptors pass an event through unchanged when they cannot parse it, so a broken regex silently stops the enrichment you depend on. Compare extracted header counts in the metrics, or log a sample event, to notice.

## How do you configure reliability and failover in Flume?

Reliability is a combination of source, channel, and sink settings. Use a durable channel, transactional sinks, and either a failover or load-balancing sink processor when more than one destination exists.

```properties
# Durable channel sized for a downstream outage
a1.channels.c1.type = file
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.checkpointDir = /data/flume/checkpoint
a1.channels.c1.capacity = 5000000
a1.channels.c1.transactionCapacity = 20000
a1.channels.c1.keep-alive = 5
```


Practical rules:

+ Set `transactionCapacity` well below `capacity` (for example 1% to 5%) so a large batch cannot starve checkpointing.
+ Monitor `ChannelFillPercentage`; a filling channel is the leading indicator of a slow sink.
+ Prefer at-least-once delivery plus idempotent downstream writes over chasing exactly-once in Flume.
+ For multi-hop agents, use Avro links and keep a durable channel on both the edge and the collector so a collector restart does not lose edge events.

Reliability is three settings: a durable channel, a transactional sink, and a sink group when there is more than one destination. `capacity` sets the outage window, `transactionCapacity` sets the batch size, and `maxpenalty` plus `backoff` control how long a failed sink is skipped before it is retried.

Worked sizing. A 4 KB event at 10,000 events/s for a 30 minute outage needs 10000 x 1800 x 4 KB, which is about 72 GB per channel. A 5 GB channel covers only about two minutes at that rate. State the outage you promise and size to it rather than discovering the limit during an incident.

Common mistake: relying on a failover sink group to handle a poison record. Failover protects against a dead destination, not against an event that always fails to parse or write, which blocks the batch forever. Route a bounded number of retries to a dead-letter channel and then move on.

## How do you monitor and tune a Flume agent?

Flume exposes a JSON metrics endpoint that can be scraped by any monitoring system, plus optional Ganglia reporting.

```bash
# Start the agent with the HTTP reporting endpoint
flume-ng agent --conf /etc/flume/conf --conf-file /etc/flume/conf/weblogs.conf -n a1 \
  -Dflume.monitoring.type=http \
  -Dflume.monitoring.port=34545 \
  -Dflume.root.logger=INFO,console
```

```bash
# Inspect channel fill and sink throughput
curl -s http://localhost:34545/metrics | python -m json.tool
```

Metrics worth alerting on:

+ `CHANNEL.c1.ChannelFillPercentage` - approaching 100% means the sink cannot keep up.
+ `CHANNEL.c1.EventPutSuccessCount` vs `EventTakeSuccessCount` - a widening gap is the same signal.
+ `SINK.k1.BatchCompleteCount` and `EventDrainSuccessCount` - throughput and stalls.
+ JVM heap and GC - a memory channel lives on heap, so a large channel needs a large heap.

The HTTP reporting endpoint exposes counters per source, channel, and sink. The three to watch are `ChannelFillPercentage`, the gap between `EventPutSuccessCount` and `EventTakeSuccessCount`, and sink health through `EventDrainSuccessCount` and `BatchCompleteCount`.

Worked alert thresholds: fill above 70% for five minutes, a put-minus-take gap that widens for ten minutes, and a sink `EventDrainSuccessCount` that stops moving while puts continue. Add JVM heap and GC pause, because a memory channel lives entirely on heap and a large channel needs a large heap.

Common mistake: monitoring only the sink. A channel near 100% with a sink that looks alive often means the sink is failing transactions and rolling back, so puts succeed while takes never commit. Compare `EventTakeAttemptCount` with `EventTakeSuccessCount` to expose rollbacks.

## How does Flume compare with Kafka Connect, Fluentd, and Filebeat?

| Tool | Model | Strength | Weakness |
|---|---|---|---|
| Flume | agent: source-channel-sink | durable HDFS/HBase sinks, mature Hadoop integration | JVM per agent, operational overhead, smaller ecosystem |
| Kafka Connect | workers plus connectors | declarative connectors, offsets managed by Kafka, huge connector library | needs Kafka as the backbone |
| Fluentd / Fluent Bit | plugin-driven collector | lightweight, cloud-native, strong filtering and parsers | HDFS/HBase sinks less common |
| Filebeat | shipper plus Logstash | tiny footprint on hosts, great for rotation-aware tailing | delivery and routing handled by a downstream tier |

Choose Flume when the destination is HDFS or HBase and the environment is already Hadoop-centric. Choose Kafka Connect or Fluent Bit for a Kafka-centric or Kubernetes platform.

The distinction that matters is where progress lives. Kafka Connect stores connector offsets in Kafka and is declarative. Flume stores progress in its own channel. Fluent Bit stores a position file and is built to be tiny. That choice decides how each tool recovers after a crash and who owns the offset.

Worked choice. Five hundred hosts at 200 KB/s into HDFS: Flume file channels hold about 100 MB per host and write to HDFS natively. The same fleet into Kafka with forty consumers is a Kafka Connect or Fluent Bit job, because HDFS is no longer the contract and the offset belongs in Kafka.

Common mistake: choosing by popularity rather than by destination. When the sink is HDFS or HBase, Flume's sink and durable file channel are still the shortest path. When the sink is Kafka, a lighter connector avoids running and operating a JVM on every host.

## When should you not use Flume?

+ You need an event backbone with replay, many consumers, and long retention - use Kafka and consume with your own application or Kafka Connect.
+ You need per-event stream processing with state, windows, and joins - use Flink or Spark Structured Streaming.
+ You need simple, host-local log shipping with low resource use - use Fluent Bit or Filebeat.
+ You need database change capture from an operational system - use Debezium or a managed CDC service rather than a Flume SQL source.
+ You need exactly-once semantics into a lakehouse - land in Kafka and use a transactional sink (Iceberg or Delta).

Flume still fits a narrow, valuable role: durable, back-pressured collection of file and syslog-style events into HDFS or HBase on an existing Hadoop cluster.

Flume is the wrong tool when the problem is compute rather than transport, when replay is required, when a host cannot afford a JVM, or when the source is a database that needs log-based change capture. It is also wrong for coordinated sub-second processing across many events.

Worked example. Five million events/s across a fleet at 100 KB per event is 500 GB/s. Per-host file channels and fsync cannot absorb that, and there is no shared retention to replay from. Kafka across many partitions plus a stream processor is the architecture; with Flume, the agents would spend their time fsyncing.

Common mistake: keeping Flume because it already runs after the destination moved to a cloud warehouse. Flume's advantage is HDFS and HBase locality. Once the destination is object storage or a cloud warehouse, a managed ingestion service usually removes the agent fleet entirely.

## What are the similarities and differences between Apache Flume and Apache Kafka?

Both move large event streams reliably and both support multiple consumers of the same data through a publish/subscribe style design.

Differences:

+ Flume pushes events to the destination through its sinks; Kafka is a durable log that consumers pull from at their own pace.
+ Flume's channel is internal to an agent and bounded by the agent's disk or heap; Kafka retains events on brokers for a configured retention period and supports replay.
+ Flume is typically an edge collection and delivery tier; Kafka is a shared backbone with many producers and consumers.
+ Flume has first-class HDFS and HBase sinks; Kafka needs a consumer, Kafka Connect, or a stream processor to reach those sinks.

They are often combined: Flume for collection and back-pressure close to the source, Kafka as the central, replayable buffer, and Connect or Flink for delivery into storage.

Both are distributed, both move events reliably, and both can feed several destinations. The fundamental difference is push versus pull. Flume removes an event from its channel once a sink has delivered it; Kafka keeps the event on the broker until the retention window expires, so any consumer can re-read it.

Worked consequence. A Flume channel holds at most `capacity` events, and one slow sink blocks every consumer of that agent because there is one queue. A Kafka topic holds a retention window, and a slow consumer group lags without affecting the others. That is why Flume sits at the edge and Kafka sits in the middle.

Common mistake: describing Flume as "Kafka without retention". Flume also lacks broker-side replication of the channel, partition rebalancing, and cluster consensus. Those absences, not just the retention window, are the real tradeoffs to state in an interview.

## Explain Reliability and Failure Handling in Apache Flume?

Flume NG uses channel-based transactions to guarantee reliable message delivery. When a message moves from one agent to another, two transactions are started: one on the agent that delivers the event and one on the agent that receives it. For the sending agent to commit its transaction, it must receive a success indication from the receiving agent. The receiving agent returns success only after its own transaction commits. This gives guaranteed at-least-once delivery between hops.

```text
Producer agent                         Consumer agent
  source -> channel                      channel -> sink
     |  begin txn                            |  begin txn
     |  write events                         |  write events
     |  commit  ----------------------------->|  commit
     |  (waits for ack)          ack          |  (returns success)
```

Failure handling:

+ If the sink fails, events stay in the channel and are retried; a file channel survives an agent restart.
+ If a downstream agent is unreachable, the upstream sink rolls back and retries, applying back-pressure to the source.
+ Duplicates are possible when an ack is lost, so sinks should be idempotent (for example, writing with a deterministic file name or an upsert).
+ A full channel blocks the source; monitoring channel fill percentage is the primary operational guardrail.

Between two agents there are two transactions, one on each side. The sending sink takes a batch, writes it to the receiving source, and commits only when the receiver's channel transaction commits and returns success. An acknowledgement lost on the way back causes a re-send, which is exactly the at-least-once boundary.

Worked failure trace. The edge sink takes 2,000 events, the collector commits them, the acknowledgement is lost, and the edge sink times out and rolls back. It re-sends the same 2,000 events, so the collector now holds 4,000 where the source produced 2,000. Duplicate handling is therefore a downstream responsibility.

Common mistake: assuming the two transactions commit atomically together. They do not; there is a window where the receiver has committed and the sender has not. Any design that needs exactly once must add an idempotency key or a deduplication step on the destination.

## How do you tune a Flume agent?

Tuning an agent is choosing where the bottleneck is allowed to live. A channel is a fixed-size buffer, so if the sink is slower than the source for longer than the buffer lasts, the source must slow down or the buffer overflows. Every tuning decision changes one of three things: the size of the buffer, the rate the sink can drain, or the size of the batch a transaction commits.

Three numbers work together. `capacity` is how many events can sit in the channel. `transactionCapacity` is the largest batch a source or sink may commit at once. Heap is the hard limit behind that: a memory channel must hold `capacity` events in the JVM, plus the in-flight source and sink batches. A common starting ratio is `transactionCapacity` at 1% to 5% of `capacity`, and a memory channel that needs 6 GB of payload needs far more than a default 1 GB heap.

Worked file-channel sizing. Event rate 30,000/s, average event 1.2 KB, outage to survive 20 minutes. The rate is 30000 x 1200 = 36,000,000 bytes/s, or 36 MB/s. Twenty minutes is 1200 seconds, so the channel must hold 36 MB/s x 1200 s = 43.2 GB. Divided by 1.2 KB per event that is 36,000,000 events, so set `capacity = 40000000` for headroom. A `transactionCapacity` of 100,000 keeps each commit at about 120 MB.

```properties
a1.channels.c1.type = file
a1.channels.c1.capacity = 40000000
a1.channels.c1.transactionCapacity = 100000
a1.channels.c1.checkpointInterval = 30000
a1.channels.c1.checkpointDir = /data/flume/checkpoint
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.fsyncPerTransaction = true
```

Reliability versus throughput is the same dial seen from two sides. `fsyncPerTransaction = true` makes every commit durable and caps throughput at the disk's fsync rate; `false` raises throughput and can lose the last few transactions on a power failure. A memory channel maximizes throughput and loses all buffered events on restart. Decide by event value: audit and billing keep `true`, rebuildable telemetry can use `false` or memory.

Common mistake: sizing `capacity` from memory instead of arithmetic. A channel set to 5,000,000 events looks generous until the average event is 4 KB, at which point it is 20 GB of payload and the agent dies on heap or disk. Compute events x average size first, then pick the channel type and the disk or heap that can actually hold it.

## How do you tune Flume batching and sink throughput?

Batch size is the main throughput lever because every transaction has a fixed cost: an RPC for Avro, a produce call for Kafka, a file append and an fsync for HDFS. Larger batches amortize that cost across more events. `batchSize` on the sink, `transactionCapacity` on the channel, and `batchSize` on the source are three names for the same batch on the three sides, and the smallest of them wins.

Worked throughput. An HDFS sink writing 1 KB events with an effective batch of 1,000 reaches 40 transactions/s, which is 40 MB/s. Raise the effective batch to 20,000 events and the same 40 MB/s needs only 2 transactions/s, a twentyfold cut in commit overhead. The bottleneck then moves to disk bandwidth and the HDFS client rather than to transaction cost.

```properties
a1.sources.r1.batchSize = 20000
a1.channels.c1.transactionCapacity = 20000

a1.sinks.k1.batchSize = 20000
a1.sinks.k1.hdfs.batchSize = 20000
a1.sinks.k1.hdfs.rollSize = 268435456
a1.sinks.k1.hdfs.rollInterval = 300
a1.sinks.k1.hdfs.rollCount = 0
```

The file channel is dominated by fsync and disk layout. Put `dataDirs` on high-IOPS storage, spread it across several directories on different devices so writes parallelize, and keep `checkpointDir` on a separate device from `dataDirs` so a checkpoint flush does not contend with data writes. `checkpointInterval` controls how much is replayed after an unclean shutdown, and `fsyncPerTransaction = false` trades the last transactions of a crash for throughput.

Common mistake: raising the sink `batchSize` without raising `transactionCapacity`. The sink asks for 20,000 events but the channel can only commit 5,000, so throughput stays flat while the sink logs channel exceptions or silently drains smaller batches. Set all three batch settings together and watch the average batch size in the metrics to confirm they agree.

## How do you diagnose drops and duplicates in Flume?

Drops and duplicates look different in the counters. A drop is a put that never becomes a successful take, or a source that never put at all. A duplicate is a take that reaches the destination twice because a transaction rolled back after the destination had already accepted the data. The metrics and the destination counts separate the two.

Start from the counters. `EventPutSuccessCount` minus `EventTakeSuccessCount` is the standing backlog; if the two converge again the events were delayed, not lost. `EventTakeAttemptCount` greater than `EventTakeSuccessCount` means rollbacks, which is the duplicate signal. A source whose put count stops moving has failed or is blocked by back-pressure from a full channel.

Worked duplicate diagnosis. The destination holds 1,080,000 rows for an hour in which the source wrote 1,000,000 events. The sink log shows 1,000 transactions and 40 retries, each retry re-sending a 2,000 event batch. The duplicates are 40 x 2000 = 80,000, exactly the gap. A timeout that rolled back after the write succeeded is the cause; an idempotency key or a shorter request timeout shrinks the ambiguous window.

```bash
grep -iE "rollback|ChannelException|Unable to commit" logs/flume.log | tail -50
grep -c "Batch complete" logs/flume.log

curl -s http://localhost:34545/metrics | python -m json.tool \
  | grep -E "EventPutSuccessCount|EventTakeSuccessCount|ChannelFillPercentage"
```

Common mistake: blaming the sink for duplicates that are source re-reads. A `taildir` source without a `positionFile`, or a spooling directory whose files reappear, re-ingests the same lines on every restart. Check the source's position tracking and file counts before you touch the sink.

## Real world use cases for Flume

### 1. Web and application log aggregation into HDFS

A media company runs 400 application servers producing 3 TB of access and application logs per day. Each host runs a lightweight Flume agent that tails the active log files and forwards over Avro to a tier of six collector agents, which write partitioned files to HDFS for Hive and Spark.

Business context: the analytics team needs daily and hourly log partitions without installing a heavy agent on every host. Data volume: 3 TB/day, roughly 35k events/second at peak.

Agent design: `taildir` source on the edge, file channel at the edge sized for 30 minutes of backlog, Avro sink to the collectors; collectors use a file channel and an HDFS sink with hour partitioning.

```properties
# Collector: partition by hour, roll at 128 MB or 10 minutes
col.sinks.k1.type = hdfs
col.sinks.k1.hdfs.path = hdfs://nn/logs/app/year=%Y/month=%m/day=%d/hour=%H
col.sinks.k1.hdfs.filePrefix = app-
col.sinks.k1.hdfs.fileType = DataStream
col.sinks.k1.hdfs.rollSize = 134217728
col.sinks.k1.hdfs.rollCount = 0
col.sinks.k1.hdfs.rollInterval = 600
col.sinks.k1.hdfs.round = true
col.sinks.k1.hdfs.roundValue = 10
col.sinks.k1.hdfs.roundUnit = minute
```

Why this design: two hops keep per-host memory small, the file channel at the edge absorbs collector restarts, and hour-level HDFS partitioning matches the Hive table's partition columns.

### 2. Clickstream into HBase for real-time lookup

An ad-tech company streams 2 billion ad impressions per day and needs millisecond lookups of recent impressions by user. Flume collects the impressions from edge servers and writes them into HBase with the user id as part of the row key.

Business context: the bidding service reads the last few impressions for a user during auction time. Data volume: ~25k events/second, 2 billion rows/day.

Agent design: a `taildir` or `http` source, a memory channel for throughput (the raw stream is also archived to S3), and an HBase sink with a regex serializer extracting `user_id`, `campaign_id`, and `ts`.

```properties
a1.sinks.k1.type = hbase
a1.sinks.k1.table = impressions
a1.sinks.k1.columnFamily = i
a1.sinks.k1.serializer = org.apache.flume.sink.hbase.RegexHbaseEventSerializer
a1.sinks.k1.serializer.regex = ^([^|]+)\\|([^|]+)\\|([^|]+)\\|(.*)$
a1.sinks.k1.serializer.colNames = user_id,campaign_id,ts,payload
a1.sinks.k1.batchSize = 500
a1.sinks.k1.channel = c2
```

Why this design: HBase gives key-based random reads at high write throughput; the row key (see the HBase topic) reverses the timestamp so the newest impressions for a user sort first. The memory channel is acceptable because a duplicate or a small loss in the ad stream is cheaper than the latency of a file channel.

### 3. Multi-datacenter log fan-in with failover

A payments platform collects audit logs from three regions into a central HDFS archive. Each region has a primary and a secondary collector; edge agents must not lose events when a collector is down.

Business context: audit logs are regulatory evidence and must be durable. Data volume: 400 GB/day, bursty at settlement time.

Agent design: an Avro sink group with a `failover` processor, priorities set so the in-region collector is preferred, and a file channel with a multi-hour capacity on every edge agent.

```properties
edge.sinkgroups = g1
edge.sinkgroups.g1.sinks = k_primary k_secondary
edge.sinkgroups.g1.processor.type = failover
edge.sinkgroups.g1.processor.priority.k_primary = 10
edge.sinkgroups.g1.processor.priority.k_secondary = 5
edge.sinkgroups.g1.processor.maxpenalty = 120000
```

Why this design: failover preserves ordering per agent better than load balancing, the file channel survives an agent restart, and the two collectors can be in different racks or regions.

### 4. IoT sensor telemetry from the edge

A utility monitors 60,000 smart meters, each emitting a reading every 15 seconds. A gateway host per substation runs Flume to buffer and forward telemetry to a central platform.

Business context: readings drive outage detection and billing reconciliation. Data volume: ~4,000 events/second, small JSON bodies.

Agent design: a `syslog` or `netcat` source on the gateway, a spillable memory channel to survive short collector outages without a large disk footprint, and a Kafka sink so multiple downstream consumers (billing, monitoring, ML) can read the stream.

```properties
gw.channels.c1.type = spillablememorychannel
gw.channels.c1.capacity = 500000
gw.channels.c1.overflowCapacity = 2000000
gw.channels.c1.checkpointDir = /data/flume/spill/checkpoint
gw.channels.c1.dataDirs = /data/flume/spill/data
```


Why this design: the spillable channel handles bursty meter traffic with a disk safety net, and publishing to Kafka lets independent teams consume the same telemetry without coupling to the collection agent.

### 5. Security and audit log archival with zero loss

A bank must retain authentication and privilege-escalation logs for seven years. Agents run on domain controllers and application servers, and evidence of loss must be avoidable.

Business context: compliance requires an immutable, complete archive. Data volume: 80 GB/day but with a strict zero-loss requirement.

Agent design: a `taildir` source with `positionFile` for restart-safe tailing, a file channel with checkpointing, and an HDFS sink writing to a WORM-style directory with zstd or gzip compression and file names that include the collector host.

```properties
a1.sources.r1.type = taildir
a1.sources.r1.positionFile = /var/lib/flume/audit_position.json
a1.sources.r1.filegroups = audit
a1.sources.r1.filegroups.audit = /var/log/audit/.*\.log
```


Why this design: `taildir` with a position file never skips a line across restarts, the file channel turns a downstream outage into back-pressure instead of loss, and per-host file prefixes make it obvious when a host is missing from the archive.

## Flume exercises

These exercises use only the Python standard library and model the tuning arithmetic from the sections above. Each one names the task, gives a hint, states the expected result, and then asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The units match what Flume uses: events per second for rate, bytes for size, and events for channel capacity. Treat the asserted numbers as a check on the formula rather than a prediction of a real agent, because production adds serialization overhead, disk variance, and retries. The value is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Size a memory channel from event rate and buffer time

**Task.** An agent receives 10,000 events/s, each about 1 KB, and must buffer 10 minutes without blocking the source. Compute the number of events, the raw payload size, and the heap to reserve if object overhead and in-flight batches roughly double the footprint.

**Hint.** `events = rate x seconds`, `bytes = events x 1024`, and `GiB = bytes / 1024**3`. Reserve twice the raw payload.

**Expected result.** `6,000,000` events, `5.72` GiB of raw payload, and about `11.44` GiB of heap.

In [1]:
EVENT_BYTES = 1024
rate_per_s = 10_000
buffer_seconds = 600

events = rate_per_s * buffer_seconds
raw_bytes = events * EVENT_BYTES
raw_gib = raw_bytes / 1024 ** 3
heap_gib = raw_gib * 2

print(f"capacity      = {events:,} events")
print(f"raw payload   = {raw_bytes:,} bytes = {raw_gib:.2f} GiB")
print(f"heap to plan  = {heap_gib:.2f} GiB")

assert events == 6_000_000
assert round(raw_gib, 2) == 5.72
assert round(heap_gib, 2) == 11.44


capacity      = 6,000,000 events
raw payload   = 6,144,000,000 bytes = 5.72 GiB
heap to plan  = 11.44 GiB


### Try it 2. Choose a batch size from sink throughput

**Task.** The sink can sustain 40 MiB/s and each event is 2 KiB. To hold commits to 20 transactions per second, compute the events per second and the batch size, then round it up to a power of two.

**Hint.** `events_per_s = sink_bytes_per_s / event_bytes`; divide by the transaction target, then round up with `1 << (n - 1).bit_length()`.

**Expected result.** `20,480` events/s and a batch of `1,024` events, which is already a power of two.

In [2]:
sink_bytes_per_s = 40 * 1024 * 1024
event_bytes = 2048
max_transactions_per_s = 20

events_per_s = sink_bytes_per_s / event_bytes
batch = events_per_s / max_transactions_per_s
batch_pow2 = 1 << (int(batch) - 1).bit_length()

print(f"sink rate        = {events_per_s:,.0f} events/s")
print(f"batch            = {batch:,.0f} events")
print(f"batch (power of 2) = {batch_pow2}")

assert events_per_s == 20480.0
assert batch == 1024.0
assert batch_pow2 == 1024


sink rate        = 20,480 events/s
batch            = 1,024 events
batch (power of 2) = 1024


### Try it 3. Compare file and memory channels for durability

**Task.** An agent receives 20,000 events/s. Its memory channel can hold 5,000,000 events and its file channel can commit 20,000 events per transaction. Compute the seconds of data at risk for each, then pick the more durable option.

**Hint.** The memory channel risks everything it holds; the file channel risks only the in-flight batch, because committed transactions survive a restart.

**Expected result.** Memory risks `250.0` seconds; the file channel risks `1.0` second, so the file channel wins.

In [3]:
rate_per_s = 20_000
memory_capacity = 5_000_000
file_transaction_capacity = 20_000

memory_loss_s = memory_capacity / rate_per_s
file_loss_s = file_transaction_capacity / rate_per_s
winner = "file channel" if file_loss_s < memory_loss_s else "memory channel"

print(f"memory channel risks {memory_loss_s:.1f} s of data")
print(f"file channel risks   {file_loss_s:.1f} s of data")
print("more durable:", winner)

assert memory_loss_s == 250.0
assert file_loss_s == 1.0
assert winner == "file channel"


memory channel risks 250.0 s of data
file channel risks   1.0 s of data
more durable: file channel


### Try it 4. Compute duplicates under failure

**Task.** One million sink transactions each carry 5,000 events, and 0.1 percent of them roll back after the destination already accepted the write. Compute the rolled-back transactions, the duplicate events, and the duplicate ratio.

**Hint.** `rolled_back = transactions x rollback_rate`; every rolled-back transaction re-sends its whole batch.

**Expected result.** `1,000` rollbacks, `5,000,000` duplicate events, a ratio of `0.001`.

In [4]:
transactions = 1_000_000
rollback_rate = 0.001
batch_events = 5_000

rolled_back = int(transactions * rollback_rate)
duplicates = rolled_back * batch_events
total_events = transactions * batch_events
duplicate_ratio = duplicates / total_events

print(f"rollbacks        = {rolled_back:,}")
print(f"duplicate events = {duplicates:,}")
print(f"duplicate ratio  = {duplicate_ratio:.4f}")

assert rolled_back == 1_000
assert duplicates == 5_000_000
assert round(duplicate_ratio, 4) == 0.001


rollbacks        = 1,000
duplicate events = 5,000,000
duplicate ratio  = 0.0010


### Try it 5. Size HDFS files from roll interval and size

**Task.** A stream writes 35 MiB/s. Compute the `rollSize` needed to produce 30 files per hour, then compute how many files per hour a 128 MiB `rollSize` would actually create.

**Hint.** `bytes_per_hour = rate x 3600`; `rollSize = bytes_per_hour / files_per_hour`.

**Expected result.** A roll size of `4200.0` MiB gives 30 files per hour, while a 128 MiB roll size gives about `984.38` files per hour, which is the small-file problem.

In [5]:
rate_mib_per_s = 35
target_files_per_hour = 30
given_roll_size_mib = 128

bytes_per_hour_mib = rate_mib_per_s * 3600
roll_size_mib = bytes_per_hour_mib / target_files_per_hour
files_per_hour_given = bytes_per_hour_mib / given_roll_size_mib

print(f"bytes per hour      = {bytes_per_hour_mib:,.0f} MiB")
print(f"rollSize for 30/h   = {roll_size_mib:,.1f} MiB")
print(f"128 MiB rollSize    = {files_per_hour_given:,.2f} files/h")

assert roll_size_mib == 4200.0
assert round(files_per_hour_given, 2) == 984.38


bytes per hour      = 126,000 MiB
rollSize for 30/h   = 4,200.0 MiB
128 MiB rollSize    = 984.38 files/h


### Try it 6. Diagnose a channel from fill percentage

**Task.** A channel with `capacity = 350,000` events recorded 1,500,000 successful puts and 1,200,000 successful takes since the agent started. Compute the standing backlog, the fill percentage, and decide whether the sink is the bottleneck.

**Hint.** `backlog = put - take` and `fill = backlog / capacity`. Treat fill above 70 percent as a sink bottleneck.

**Expected result.** A backlog of `300,000` events, `85.7` percent fill, and the verdict that the sink is the bottleneck.

In [6]:
capacity = 350_000
put_success = 1_500_000
take_success = 1_200_000

backlog = put_success - take_success
fill_pct = backlog / capacity * 100
verdict = "sink is the bottleneck" if fill_pct > 70 else "healthy"

print(f"backlog = {backlog:,} events")
print(f"fill    = {fill_pct:.1f}%")
print("verdict:", verdict)

assert backlog == 300_000
assert round(fill_pct, 1) == 85.7
assert verdict == "sink is the bottleneck"


backlog = 300,000 events
fill    = 85.7%
verdict: sink is the bottleneck


### Try it 7. Pick a transactionCapacity inside the healthy ratio

**Task.** A channel holds 2,000,000 events of 1 KiB and you want each commit to move about 50 MiB. Compute the batch in events, the ratio to `capacity`, and check whether it falls inside the recommended 1 to 5 percent band.

**Hint.** `batch_events = target_batch_mb x 1024**2 / event_bytes`; the ratio is `batch_events / capacity`.

**Expected result.** `51,200` events per batch, a ratio of `0.0256` (2.56 percent), which is inside the band.

In [7]:
capacity = 2_000_000
event_bytes = 1024
target_batch_mib = 50

batch_events = int(target_batch_mib * 1024 ** 2 / event_bytes)
ratio = batch_events / capacity
within_band = 0.01 <= ratio <= 0.05

print(f"transactionCapacity = {batch_events:,} events")
print(f"ratio to capacity   = {ratio:.4f}")
print("within 1-5%:", within_band)

assert batch_events == 51_200
assert round(ratio, 4) == 0.0256
assert within_band is True


transactionCapacity = 51,200 events
ratio to capacity   = 0.0256
within 1-5%: True


### Try it 8. Size a spillable channel for a burst

**Task.** A spillable memory channel buffers 500,000 events on heap and can spill another 2,000,000 to disk. At a burst of 4,000 events/s, compute how long until spilling starts and how long until the disk overflow is exhausted.

**Hint.** Divide each capacity by the burst rate; the total is the heap capacity plus the overflow capacity.

**Expected result.** Spilling starts after `125.0` seconds and the overflow is exhausted after `625.0` seconds.

In [8]:
heap_capacity = 500_000
overflow_capacity = 2_000_000
burst_rate = 4_000

spill_after_s = heap_capacity / burst_rate
disk_after_s = (heap_capacity + overflow_capacity) / burst_rate

print(f"spill starts after      {spill_after_s:.1f} s")
print(f"disk overflow exhausted {disk_after_s:.1f} s")

assert spill_after_s == 125.0
assert disk_after_s == 625.0


spill starts after      125.0 s
disk overflow exhausted 625.0 s
